# WAV1 mechanism factorization — ALL-IN-ONE Kaggle
Satu notebook untuk seluruh Stage-1 mechanistic screen: preflight → clone repo → validate existing Faruq core → static audit → `HP1 → WAV_L1 → WAV_L2 → WAV_RAWFUSE` → mechanistic report → ZIP.

Tidak ada Colab add-on. Tidak ada retraining `WAV1_REF`. Locked test tetap tertutup.

**Input Kaggle yang diperlukan hanya private dataset existing:** `faruq-v3-experiment-core-v1`.


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
core=sorted(INPUT.rglob('af2_spectral_kaggle_manifest.json'))
if len(core)!=1: raise FileNotFoundError(f'Harus ada tepat satu core manifest; ditemukan {core}')
print('INPUT PREFLIGHT PASS')
print('CORE:', core[0])


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile,torch
os.chdir(WORK)
REPO=WORK/'coffee-bean-detection'
BRANCH='agent/wav1-mechanism-factorization'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('COMMIT:',COMMIT)
print('ULTRALYTICS:',__import__('ultralytics').__version__)


In [ ]:
# Validate the already-existing private Faruq core. No new Kaggle dataset/add-on is needed.
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
DATA,ARTIFACTS,CORE_CONTRACT=prepare_af2_spectral_kaggle_input(INPUT,WORK)
assert CORE_CONTRACT['test_images_accessed'] is False
D0=ARTIFACTS['D0_seed42_best.pt']
print('CORE CONTRACT PASS')
print('DATA:',DATA)
print('D0:',D0)

# Reuse frozen D0FT seed42 control from the existing breadth-screen artifact in the core.
breadth=json.loads(Path(ARTIFACTS['lfdet_afab_seed42_screening.json']).read_text(encoding='utf-8'))
assert breadth['protocol']=='faruq-v3-lfdet-afab-breadth-screening-v1'
assert int(breadth['seed'])==42 and breadth['evaluation_split']=='val'
assert breadth['test_images_accessed'] is False and breadth['test_opened'] is False
d0ft_payload={
    'format':'coffee_detector.wav1_factorization.reference.v1',
    'arm':'D0FT','seed':42,
    'metrics':breadth['controls']['D0FT'].get('metrics',breadth['controls']['D0FT']),
    'evaluation_split':'val','test_images_accessed':False
}
D0FT_REF=WORK/'d0ft_seed42_reference.json'
D0FT_REF.write_text(json.dumps(d0ft_payload,indent=2)+'\n',encoding='utf-8')

# Frozen WAV1 seed42 headline reference from the completed paired-confirmation protocol.
WAV1_FROZEN={
 'macro_map50_95':0.8841052369918866,
 'bottom3_class_map50_95':0.8327607439278027,
 'worst_class_map50_95':0.8203489485589485,
}
wav1_payload={
    'format':'coffee_detector.wav1_factorization.reference.v1',
    'arm':'WAV1_REF','seed':42,
    'metrics':WAV1_FROZEN,
    'evaluation_split':'val','test_images_accessed':False,
    'note':'Frozen seed42 headline metrics from completed WAV1 paired confirmation; no WAV1 retraining in this notebook.'
}
WAV1_REF=WORK/'wav1_seed42_frozen_reference.json'
WAV1_REF.write_text(json.dumps(wav1_payload,indent=2)+'\n',encoding='utf-8')
print('REFERENCES READY:',D0FT_REF,WAV1_REF)


In [ ]:
# Contract tests + static audit before any training.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_wav1_factorization.py'],cwd=REPO,check=True)
from coffee_detector.wav1_factorization.audit import run_static_audit
OUT=WORK/'wav1-factorization-stage1-v1'; OUT.mkdir(exist_ok=True)
STATIC=OUT/'static_audit.json'
audit=run_static_audit(D0,STATIC)
print(json.dumps(audit,indent=2))
if audit['decision']!='PASS' or not audit['training_authorized']:
    raise RuntimeError('STOP: static audit gagal')
assert audit['wav1_ref_bitwise_equal_to_confirmed_operator'] is True
assert audit['test_access_authorized'] is False

# Deterministic CUDA smoke for the exact causal arms.
from coffee_detector.wav1_factorization import TRAIN_ARMS,WAV1FactorizationEnhancer,frozen_arm_config
previous=torch.are_deterministic_algorithms_enabled()
try:
    torch.use_deterministic_algorithms(True,warn_only=False)
    for arm in TRAIN_ARMS:
        x=torch.rand(1,3,65,63,device='cuda:0',requires_grad=True)
        f=WAV1FactorizationEnhancer(frozen_arm_config(arm)).to('cuda:0')
        y=f(x); y.mean().backward()
        assert torch.isfinite(y).all() and x.grad is not None and torch.isfinite(x.grad).all()
        print('CUDA SMOKE',arm,'PASS')
finally:
    torch.use_deterministic_algorithms(previous)


In [ ]:
# Train the four frozen causal arms sequentially. Snapshot after each arm.
ARMS=('HP1','WAV_L1','WAV_L2','WAV_RAWFUSE')
SNAPSHOT_BASE=WORK/'wav1-factorization-stage1-output'
def snapshot():
    p=Path(shutil.make_archive(str(SNAPSHOT_BASE),'zip',OUT)); print('SNAPSHOT:',p,flush=True); return p
def run_arm(arm):
    result=OUT/'val_reports'/f'{arm}_seed42_result.json'
    log=OUT/f'{arm}_seed42.log'
    if result.is_file():
        print('REUSE COMPLETE',arm,result); snapshot(); return
    cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_wav1_factorization_arm',
         '--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),
         '--d0-checkpoint',str(D0),'--static-audit',str(STATIC),'--output-root',str(OUT),
         '--seed','42','--device','0','--authorize-training']
    print('START',arm,flush=True)
    with log.open('a',encoding='utf-8') as stream:
        p=subprocess.Popen(cmd,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    prev=None
    while p.poll() is None:
        csv=OUT/arm/f'{arm}_seed42'/'results.csv'
        epoch=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epoch!=prev: print(f'{arm}: {epoch}/50 epoch',flush=True); prev=epoch
        time.sleep(120)
    if p.returncode:
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-160:]) if log.is_file() else '<no log>'
        raise RuntimeError(f'{arm} gagal, returncode={p.returncode}\n{tail}')
    if not result.is_file(): raise RuntimeError(f'Hasil tidak ditemukan: {result}')
    payload=json.loads(result.read_text(encoding='utf-8'))
    assert payload['evaluation_split']=='val' and payload['test_images_accessed'] is False
    print('DONE',arm,flush=True); snapshot()
for arm in ARMS: run_arm(arm)


In [ ]:
# Build the mechanistic report. Because the all-in-one notebook uses the frozen WAV1 headline reference,
# headline preservation ratios are available; per-class WAV1 delta correlation is intentionally unavailable here.
from coffee_detector.experiments.run_faruq_v3_wav1_factorization_decision import run_factorization_report
arm_results=[OUT/'val_reports'/f'{arm}_seed42_result.json' for arm in ARMS]
REPORT=OUT/'val_reports'/'wav1_factorization_seed42_report.json'
report=run_factorization_report(D0FT_REF,WAV1_REF,arm_results,REPORT)
print('=== HEADLINE MECHANISTIC SCREEN ===')
print('D0FT:',report['references']['d0ft'])
print('WAV1:',report['references']['wav1'])
for row in report['arms']:
    print('\n',row['arm'])
    print(' metrics:',row['metrics'])
    print(' gain_vs_d0ft:',row['gain_vs_d0ft'])
    print(' wav1_gain_preservation:',row['wav1_gain_preservation'])
print('\nDECISION:',report['decision'])
print('NOTE: per-class WAV1 correlation is unavailable in this simplified notebook because the full frozen WAV1 per-class JSON is not required as input.')
archive=snapshot()
print('FINAL ZIP:',archive)
print('REPORT:',REPORT)
print('STOP HERE. Jangan buka seed 123/2026 atau locked test sebelum hasil ini direview.')
